# Keyword Extraction from Redacted Text with YAKE! on SageMaker

Follows the same **`PySparkProcessor`** pattern as `redaction.ipynb`:

1. Write `key_ana_script.py` — the PySpark + Spark-NLP script that runs *inside* the managed container
2. Resolve IAM role and stage the Spark-NLP assembly JAR on S3
3. Submit a SageMaker Processing Job — SageMaker provisions the instance, mounts inputs, runs the script, copies output to S3, then **automatically tears down the instance**

| | Value |
|---|---|
| **Input parquet** | `s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/redacted_ana/redacted_ana.parquet` |
| **Output parquet** | `s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/key_ana/` |
| **New column** | `yake_keywords` — YAKE! keywords extracted from `redacted_text` |

**Pipeline:**

| Step | Component | Purpose |
|---|---|---|
| 1 | `DocumentAssembler` | Wrap raw text |
| 2 | `SentenceDetector` | Segment into sentences |
| 3 | `Tokenizer` | Tokenise |
| 4 | `YakeKeywordExtraction` | Extract statistical keywords |

## 1. Environment Setup

In [1]:
import subprocess, sys, os

java_check = subprocess.run(["java", "-version"], capture_output=True)
if java_check.returncode != 0:
    print("Java not found — installing default-jdk ...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "default-jdk"], check=True)
    print("Java installed.")
else:
    print("Java already available:", java_check.stderr.decode().split('\n')[0])

java_home_candidates = [
    "/usr/lib/jvm/java-11-openjdk-amd64",
    "/usr/lib/jvm/java-11-amazon-corretto",
    "/usr/lib/jvm/java-8-openjdk-amd64",
    "/usr/lib/jvm/java-8-amazon-corretto",
]
for candidate in java_home_candidates:
    if os.path.isdir(candidate):
        os.environ["JAVA_HOME"] = candidate
        break
else:
    result = subprocess.run(
        ["java", "-XshowSettings:all", "-version"],
        capture_output=True, text=True
    )
    for line in result.stderr.splitlines():
        if "java.home" in line:
            java_home = line.split("=")[1].strip()
            if java_home.endswith("/jre"):
                java_home = java_home[:-4]
            os.environ["JAVA_HOME"] = java_home
            break

print("JAVA_HOME:", os.environ.get("JAVA_HOME", "NOT SET"))

Java already available: openjdk version "11.0.30" 2026-01-20
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [2]:
!pip install -q \
    "sagemaker>=2.0,<3.0" \
    "pyspark>=3.3,<3.6" \
    "spark-nlp==5.5.3" \
    "boto3"

print("All packages installed.")

All packages installed.


## 2. SageMaker v2.x Session Setup

In [3]:
import boto3
import sagemaker
from sagemaker import get_execution_role

boto_session = boto3.Session()
sm_session   = sagemaker.Session(boto_session=boto_session)

try:
    role = get_execution_role(sagemaker_session=sm_session)
except Exception:
    role = os.environ.get("SAGEMAKER_ROLE", "<YOUR_SAGEMAKER_EXECUTION_ROLE_ARN>")
    print("WARNING: Could not auto-detect execution role. Set SAGEMAKER_ROLE env var.")

region  = boto_session.region_name or sm_session.boto_region_name
account = boto3.client("sts").get_caller_identity()["Account"]

# Fixed bucket and paths
BUCKET    = "sagemaker-us-west-2-493644444178"
prefix    = "spark-nlp/key-ana"               # staging area for JAR

INPUT_S3  = f"s3://{BUCKET}/spark-nlp/redaction/jgh-msss-prem/redacted_output/redacted_ana"
OUTPUT_S3 = f"s3://{BUCKET}/spark-nlp/redaction/jgh-msss-prem/redacted_output/key_ana"

print(f"SageMaker SDK version : {sagemaker.__version__}")
print(f"Region                : {region}")
print(f"Execution role ARN    : {role}")
print(f"Input  S3             : {INPUT_S3}")
print(f"Output S3             : {OUTPUT_S3}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/cjen/.config/sagemaker/config.yaml
SageMaker SDK version : 2.257.1
Region                : us-west-2
Execution role ARN    : arn:aws:iam::493644444178:role/aws-reserved/sso.amazonaws.com/ap-southeast-1/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd
Input  S3             : s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/redacted_ana
Output S3             : s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/key_ana


## 3. Write PySpark Processing Script

This script runs **inside the SageMaker container** (`spark-submit`).
It reads the mounted input parquet, applies the YAKE! pipeline,
adds the `yake_keywords` column, and writes the result to the output mount.

> YAKE! (`YakeKeywordExtraction`) is a purely statistical extractor — no
> pretrained model files are required, only the Spark-NLP assembly JAR.

In [4]:
%%writefile key_ana_script.py
"""
Spark-NLP YAKE! keyword extraction processing script.
Executed inside a SageMaker PySparkProcessor container via spark-submit.

Processes two redacted text columns:
  - redacted_aspects      -> yake_keywords_aspects
  - redacted_suggestions  -> yake_keywords_suggestions
"""
import os, sys, subprocess, glob, shutil, traceback

# ── Early diagnostics ─────────────────────────────────────────────────────────
print("=" * 60, flush=True)
print(f"Python  : {sys.version}", flush=True)
print(f"CWD     : {os.getcwd()}", flush=True)
print("=" * 60, flush=True)

# ── Install Python-side spark-nlp bindings (JAR loaded via --jars) ────────────
pip_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "spark-nlp==5.5.3", "pandas<2.0"],
    capture_output=True, text=True,
)
print(f"pip exit={pip_result.returncode}", flush=True)
if pip_result.returncode != 0:
    print("pip stderr:", pip_result.stderr[-3000:], flush=True)

try:
    import sparknlp
    from sparknlp.base import DocumentAssembler
    from sparknlp.annotator import SentenceDetector, Tokenizer, YakeKeywordExtraction
    from pyspark.ml import Pipeline
    from pyspark.sql import SparkSession
    import pyspark.sql.functions as F
    print(f"Imports OK — spark-nlp {sparknlp.version()}", flush=True)
except Exception:
    print("IMPORT ERROR:", flush=True)
    traceback.print_exc()
    sys.exit(1)

INPUT_DIR  = "/opt/ml/processing/input"
OUTPUT_DIR = "/opt/ml/processing/output"

# ── Start Spark ───────────────────────────────────────────────────────────────
try:
    spark = SparkSession.builder.appName("SparkNLP-KeyAna").getOrCreate()
    print(f"Spark {spark.version} started", flush=True)
except Exception:
    print("SPARKSESSION ERROR:", flush=True)
    traceback.print_exc()
    sys.exit(1)

# ── Read input parquet ────────────────────────────────────────────────────────
parquet_files = (
    glob.glob(os.path.join(INPUT_DIR, "**/*.parquet"), recursive=True)
    + glob.glob(os.path.join(INPUT_DIR, "*.parquet"))
)
if not parquet_files:
    raise FileNotFoundError(f"No parquet files found under {INPUT_DIR}")

print(f"Reading parquet from: {INPUT_DIR}", flush=True)
spark_df = spark.read.parquet("file://" + INPUT_DIR)
print(f"Input  : {spark_df.count()} rows × {len(spark_df.columns)} columns", flush=True)
spark_df.printSchema()

# ── Columns to extract keywords from ──────────────────────────────────────────
TEXT_COLS = {
    "redacted_aspects":     "yake_keywords_aspects",
    "redacted_suggestions": "yake_keywords_suggestions",
}
missing = [c for c in TEXT_COLS if c not in spark_df.columns]
if missing:
    raise ValueError(
        f"Expected columns {list(TEXT_COLS)} not found. "
        f"Available: {spark_df.columns}"
    )
print(f"Will extract keywords from: {list(TEXT_COLS.keys())}", flush=True)

# ── Build YAKE! pipeline ──────────────────────────────────────────────────────
def build_yake_pipeline(input_col: str) -> Pipeline:
    doc = (
        DocumentAssembler()
        .setInputCol(input_col)
        .setOutputCol("document")
    )
    sent = (
        SentenceDetector()
        .setInputCols(["document"])
        .setOutputCol("sentence")
    )
    tok = (
        Tokenizer()
        .setInputCols(["sentence"])
        .setOutputCol("token")
    )
    yake = (
        YakeKeywordExtraction()
        .setInputCols(["token"])
        .setOutputCol("keywords")
        .setMinNGrams(1)
        .setMaxNGrams(3)
        .setNKeywords(30)
        .setThreshold(1.5)
        .setWindowSize(3)
    )
    return Pipeline(stages=[doc, sent, tok, yake])

result_df = spark_df
for src_col, out_col in TEXT_COLS.items():
    print(f"  Processing column: {src_col} -> {out_col}", flush=True)
    working_df = result_df.withColumn("_text_input", F.col(src_col))
    pipeline = build_yake_pipeline("_text_input")
    empty_fit = spark.createDataFrame([[""]], ["_text_input"])
    model = pipeline.fit(empty_fit)
    transformed = model.transform(working_df)
    result_df = transformed.withColumn(
        out_col, F.array_distinct(F.col("keywords.result"))
    ).drop("document", "sentence", "token", "keywords", "_text_input")
    print(f"    Done.", flush=True)

# ── Select original columns + new keyword columns ─────────────────────────────
output_cols = spark_df.columns + list(TEXT_COLS.values())
output_df   = result_df.select(output_cols)

print(f"Output : {output_df.count()} rows, columns: {output_df.columns}", flush=True)

# ── Write output parquet ──────────────────────────────────────────────────────
# The SageMaker bind-mount at /opt/ml/processing/output blocks Java FileSystem
# delete operations (including Spark's overwrite-mode pre-delete). Work around:
# 1. Write to /tmp (unrestricted) via Spark.
# 2. Copy the resulting parquet files to the output mount using Python shutil.
# SageMaker will then upload /opt/ml/processing/output/data -> OUTPUT_S3.

TMP_PATH = "/tmp/spark_output"
if os.path.exists(TMP_PATH):
    shutil.rmtree(TMP_PATH)

output_df.coalesce(1).write.mode("overwrite").parquet("file://" + TMP_PATH)
print(f"Written to tmp: {TMP_PATH}", flush=True)

out_dir = os.path.join(OUTPUT_DIR, "data")
os.makedirs(out_dir, exist_ok=True)
for fname in os.listdir(TMP_PATH):
    src = os.path.join(TMP_PATH, fname)
    dst = os.path.join(out_dir, fname)
    shutil.copy2(src, dst)
    print(f"  Copied {fname} -> {dst}", flush=True)

print(f"Output ready at: {out_dir}", flush=True)

spark.stop()
print("Done.", flush=True)


Overwriting key_ana_script.py


## 4. Resolve SageMaker-Assumable IAM Role

SSO roles (`aws-reserved/sso.amazonaws.com/...`) cannot be assumed by
`sagemaker.amazonaws.com`. This cell finds or creates a dedicated role.

In [5]:
import boto3, json, time

iam          = boto3.client("iam")
SM_ROLE_NAME = "SageMakerExecutionRole-SparkNLP"

TRUST_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect"   : "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action"   : "sts:AssumeRole",
    }],
})

try:
    sm_role = iam.get_role(RoleName=SM_ROLE_NAME)["Role"]["Arn"]
    print(f"Reusing existing role: {sm_role}")

except iam.exceptions.NoSuchEntityException:
    print(f"Creating '{SM_ROLE_NAME}' ...")
    sm_role = iam.create_role(
        RoleName                 = SM_ROLE_NAME,
        AssumeRolePolicyDocument = TRUST_POLICY,
        Description              = "SageMaker execution role for Spark-NLP processing jobs",
    )["Role"]["Arn"]

    for policy_arn in [
        "arn:aws:iam::aws:policy/AmazonSageMakerFullAccess",
        "arn:aws:iam::aws:policy/AmazonS3FullAccess",
    ]:
        iam.attach_role_policy(RoleName=SM_ROLE_NAME, PolicyArn=policy_arn)
        print(f"  Attached {policy_arn}")

    print("Waiting 15 s for IAM propagation ...")
    time.sleep(15)

print(f"\nsm_role = {sm_role}")

Reusing existing role: arn:aws:iam::493644444178:role/SageMakerExecutionRole-SparkNLP

sm_role = arn:aws:iam::493644444178:role/SageMakerExecutionRole-SparkNLP


## 5. Download & Stage Spark-NLP Assembly JAR on S3

The SageMaker container has no internet access, so we pre-fetch the JAR here
(this notebook has internet) and stage it on S3.

> No pretrained model files are needed — YAKE! is purely statistical.

In [6]:
import urllib.request
from sagemaker.s3 import S3Uploader

NLP_VER  = "5.5.3"
jar_name = f"spark-nlp-assembly-{NLP_VER}.jar"
local    = f"/tmp/{jar_name}"

candidate_urls = [
    f"https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/jars/{jar_name}",
    (
        f"https://repo1.maven.org/maven2/com/johnsnowlabs/nlp/"
        f"spark-nlp-assembly_2.12/{NLP_VER}/"
        f"spark-nlp-assembly_2.12-{NLP_VER}.jar"
    ),
]

downloaded = False
for url in candidate_urls:
    try:
        print(f"Trying  {url}")
        urllib.request.urlretrieve(url, local)
        size_mb = os.path.getsize(local) / 1_048_576
        print(f"  size  {size_mb:.1f} MB")
        downloaded = True
        break
    except Exception as e:
        print(f"  ✗  {e}")

if not downloaded:
    raise RuntimeError(
        f"Could not download the spark-nlp assembly JAR.\n"
        f"Download manually from https://github.com/JohnSnowLabs/spark-nlp/releases/tag/{NLP_VER}\n"
        f"and save as /tmp/{jar_name}, then re-run from the S3Uploader line."
    )

# Validate it's actually a JAR (ZIP magic bytes)
with open(local, "rb") as f:
    magic = f.read(4)
if magic[:2] != b"PK":
    raise RuntimeError(
        f"Downloaded file is NOT a valid JAR — got header {magic!r}.\n"
        "The URL likely returned an HTML error page. Try the manual download above."
    )
print(f"JAR valid  (magic={magic!r}, size={os.path.getsize(local)/1_048_576:.0f} MB)")

dst_s3_dir      = f"s3://{BUCKET}/{prefix}/jars"
S3Uploader.upload(local, dst_s3_dir, sagemaker_session=sm_session)
sparknlp_jar_s3 = f"{dst_s3_dir}/{jar_name}"
print(f"Staged at  {sparknlp_jar_s3}")

Trying  https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/jars/spark-nlp-assembly-5.5.3.jar
  size  605.3 MB
JAR valid  (magic=b'PK\x03\x04', size=605 MB)
Staged at  s3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar


## 6. Submit SageMaker Processing Job

`PySparkProcessor` provisions an `ml.m5.4xlarge` instance, runs `key_ana_script.py`
inside the managed Spark container, copies output to S3, and **automatically
terminates the instance** when the job finishes (success or failure).

In [7]:
from sagemaker.spark.processing import PySparkProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

spark_processor = PySparkProcessor(
    base_job_name          = "spark-nlp-key-ana",
    framework_version      = "3.3",
    role                   = sm_role,
    instance_count         = 1,
    instance_type          = "ml.m5.4xlarge",   # 16 vCPU / 64 GB
    max_runtime_in_seconds = 3600,
    sagemaker_session      = sm_session,
)

spark_processor.run(
    submit_app  = "key_ana_script.py",
    submit_jars = [sparknlp_jar_s3],
    configuration = [{
        "Classification": "spark-defaults",
        "Properties": {
            "spark.serializer"                : "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max" : "2000M",
            "spark.driver.memory"             : "16G",
            "spark.driver.maxResultSize"      : "4000M",
            "spark.executor.memory"           : "20G",
            "spark.executor.memoryOverhead"   : "4G",
        },
    }],
    inputs = [
        ProcessingInput(
            source      = INPUT_S3,
            destination = "/opt/ml/processing/input",
        ),
    ],
    outputs = [
        ProcessingOutput(
            # Script writes to /opt/ml/processing/output/data (subdirectory)
            # so Spark can freely create/overwrite it without touching the
            # SageMaker-mounted root. SageMaker uploads this subdir's contents
            # directly to OUTPUT_S3 — no double key_ana/ nesting.
            source      = "/opt/ml/processing/output/data",
            destination = OUTPUT_S3,
        )
    ],
    logs = True,
    wait = True,   # blocks until job completes; instance auto-terminates after
)

print(f"\nJob complete. Results at: {OUTPUT_S3}")


INFO:sagemaker:Creating processing-job with name spark-nlp-key-ana-2026-03-28-03-55-51-613


............03-28 03:57 smspark.cli  INFO     Parsing arguments. argv: ['/usr/local/bin/smspark-submit', '--jars', 's3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar', '/opt/ml/processing/input/code/key_ana_script.py']
03-28 03:57 smspark.cli  INFO     Raw spark options before processing: {'jars': 's3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar', 'class_': None, 'py_files': None, 'files': None, 'verbose': False}
03-28 03:57 smspark.cli  INFO     App and app arguments: ['/opt/ml/processing/input/code/key_ana_script.py']
03-28 03:57 smspark.cli  INFO     Rendered spark options: {'jars': 's3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar', 'class_': None, 'py_files': None, 'files': None, 'verbose': False}
03-28 03:57 smspark.cli  INFO     Initializing processing job.
03-28 03:57 smspark-submit INFO     {'current_host': 'algo-1', 'current_instance_type': 'ml.m5.4xlarge

## 7. Fetch CloudWatch Logs

In [8]:
import boto3, time

logs      = boto3.client("logs", region_name=region)
job_name  = spark_processor.latest_job.job_name
log_group = "/aws/sagemaker/ProcessingJobs"

print(f"Job      : {job_name}")
print(f"LogGroup : {log_group}\n")

for attempt in range(10):
    resp    = logs.describe_log_streams(logGroupName=log_group, logStreamNamePrefix=job_name)
    streams = resp.get("logStreams", [])
    if streams:
        break
    print(f"  Waiting for log streams (attempt {attempt+1}/10) ...")
    time.sleep(6)

for stream in streams:
    stream_name = stream["logStreamName"]
    print(f"\n{'='*60}\nStream: {stream_name}\n{'='*60}")
    events_resp = logs.get_log_events(
        logGroupName  = log_group,
        logStreamName = stream_name,
        startFromHead = True,
    )
    for event in events_resp["events"]:
        print(event["message"])

Job      : spark-nlp-key-ana-2026-03-28-03-55-51-613
LogGroup : /aws/sagemaker/ProcessingJobs


Stream: spark-nlp-key-ana-2026-03-28-03-55-51-613/algo-1-1774670189
03-28 03:57 smspark.cli  INFO     Parsing arguments. argv: ['/usr/local/bin/smspark-submit', '--jars', 's3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar', '/opt/ml/processing/input/code/key_ana_script.py']
03-28 03:57 smspark.cli  INFO     Raw spark options before processing: {'jars': 's3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar', 'class_': None, 'py_files': None, 'files': None, 'verbose': False}
03-28 03:57 smspark.cli  INFO     App and app arguments: ['/opt/ml/processing/input/code/key_ana_script.py']
03-28 03:57 smspark.cli  INFO     Rendered spark options: {'jars': 's3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar', 'class_': None, 'py_files': None, 'files': None, 'verbose': False}
03-28 03:57 s

## 8. Preview Output from S3

In [9]:
import io, boto3, pandas as pd

s3_client  = boto3.client("s3")
out_prefix = OUTPUT_S3.replace(f"s3://{BUCKET}/", "").rstrip("/") + "/"

paginator    = boto3.client("s3").get_paginator("list_objects_v2")
output_files = [
    obj["Key"]
    for page in paginator.paginate(Bucket=BUCKET, Prefix=out_prefix)
    for obj in page.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]

print(f"Parquet files in output: {len(output_files)}")
for k in output_files:
    print(f"  s3://{BUCKET}/{k}")

if output_files:
    buf = io.BytesIO()
    s3_client.download_fileobj(BUCKET, output_files[0], buf)
    buf.seek(0)
    preview = pd.read_parquet(buf)
    print(f"\nPreview ({len(preview)} rows, columns: {list(preview.columns)}):")
    display(preview.head(10))

Parquet files in output: 2
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/key_ana/part-00000-43b6c2c9-2b0d-4606-a9b4-5a9184d03eef-c000.snappy.parquet
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/key_ana/part-00000-63dcb776-96f5-43de-90c1-f4213432611c-c000.snappy.parquet

Preview (5740 rows, columns: ['Unnamed: 0', 'Date de complétion', 'Évaluation', "Aspects positifs de l'expérience globale", 'redacted_aspects', "Suggestions d'amélioration de l'expérience globale", 'redacted_suggestions', 'yake_keywords_aspects', 'yake_keywords_suggestions']):


,Unnamed: 0,Date de complétion,Évaluation,Aspects positifs de l'expérience globale,redacted_aspects,Suggestions d'amélioration de l'expérience globale,redacted_suggestions,yake_keywords_aspects,yake_keywords_suggestions
0,1,2025-09-13 11:41:05.701000+00:00,,The doctor did a thorough review and told us w...,The doctor did a thorough review and told us w...,,,"[doctor, thorough, review, told, us, done, tho...",[]
1,4,2025-11-11 17:24:25.463000+00:00,,Very professional and knowledgable about healt...,Very professional and knowledgable about healt...,The care was excellent.,The care was excellent.,"[professional, knowledgable, health, needs, co...","[care, excellent]"
2,7,2026-01-06 15:39:26.932000+00:00,,Efficiency of the system. Overall time was 4:3...,Efficiency of the system. Overall time was 4:3...,Continue the great work,Continue the great work,"[efficiency, system, overall, time, hours, see...","[continue, great, work, great work]"
3,8,2025-06-04 03:29:18.254000+00:00,,It was too Good for being there for meet up my...,It was too Good for being there for meet up my...,,,"[good, meet, emergency, need, emergency need]",[]
4,9,2024-11-13 00:38:07.856000+00:00,,,,Please install drinking water fountains.,Please install drinking water fountains.,[],"[please, install, drinking, water, fountains, ..."
5,10,2024-08-08 02:37:59.557000+00:00,,Triage in 30 minutes,Triage in 30 minutes,My visit lasted over 10 hours . It was impossi...,My visit lasted over 10 hours . It was impossi...,"[triage, minutes]","[visit, lasted, hours, impossible, times, find..."
6,12,2024-12-04 21:31:57.278000+00:00,,Superb; impressed with devotion of staff.,Superb; impressed with devotion of staff.,More info. on discharge. There was no advice ...,More info. on discharge. There was no advice ...,"[superb, impressed, devotion, staff]","[info, discharge, advice, given, one, weak, spot]"
7,13,2026-01-13 22:46:21.866000+00:00,,Fast,Fast,"Fast, but waited a long time for chest x-ray r...","Fast, but waited a long time for chest x-ray r...",[fast],"[fast, waited, long, time, chest, results, lon..."
8,14,2024-10-24 15:42:38.865000+00:00,,I'm out of the place,I'm out of the place,Placed in room with other sick persons could f...,Placed in room with other sick persons could f...,[place],"[placed, room, sick, persons, find, place, sit..."
9,16,2025-03-25 14:06:10.159000+00:00,,"follow-up, timely appointments","follow-up, timely appointments",,,"[timely, appointments, timely appointments]",[]
